# WindShift — two independent reviewer-model replications

**Two additional model families** on the **exact 192 frozen reviewer prompts** from v3/v4: Anthropic **Claude Haiku 5.5** and an open-weight **Qwen3-235B Instruct** model. The original GPT-4o-mini and Gemini results are embedded for comparison. This notebook is fully self-contained; do **not** upload the research ZIP.

**Scientific scope:** replaces the reviewing agent only, keeping the historical GPT peer reports and all synthetic observations fixed. This is NOT full swarm replication, independent episodes, or network-size scaling. Models are chosen before results; both must be reported, including invalid or negative outcomes. New results are post-hoc cross-model follow-ups, not preregistered v3 evidence.

**Safety/cost:** All calls are OFF by default. Real API calls require explicit `AUTHORIZE_PILOT=True` or `AUTHORIZE_FULL=True` after reviewing your credits. Calls may incur fees even if they fail. No key will be written to the archive.

In [ ]:
# Cell 1 — Recover frozen data (offline, no API calls)
import os,sys,json,hashlib,base64,zlib,random,time,datetime,csv,re,urllib.request,urllib.error,zipfile,io
from pathlib import Path
ROOT=Path('/content/windshift_models') if Path('/content').exists() else Path.cwd()/'windshift_models'
ROOT.mkdir(parents=True,exist_ok=True)
PROMPT_B64='...'
GEMINI_B64='...'
PROMPT_RAW=zlib.decompress(base64.b64decode(PROMPT_B64))
GEMINI_RAW=zlib.decompress(base64.b64decode(GEMINI_B64))
(ROOT/'frozen_reviewer_prompts.json').write_bytes(PROMPT_RAW)
(ROOT/'gemini_reviewer_calls.jsonl').write_bytes(GEMINI_RAW)
PROMPTS=json.loads(PROMPT_RAW);GEMINI=[json.loads(i) for i in GEMINI_RAW.decode().splitlines() if i.strip()]
print('Frozen prompts:',len(PROMPTS),'| Gemini calls:',len(GEMINI))
print('Prompt SHA256:',hashlib.sha256(PROMPT_RAW).hexdigest())
print('No paid calls made.')

In [ ]:
# Cell 2 — Preflight, identity checks, and exact pilot sampling (offline)
ARMS=('fresh_peer','current_content_old_timestamp','historical_content_new_timestamp','stale_peer')
MODELS={
   'qwen': {'id':'qwen/qwen3-235b-a22b-2507','max_tokens':1024},
   'claude': {'id':'anthropic/claude-haiku-5.5','max_tokens':1024},
}
assert len(PROMPTS)==192 and len(GEMINI)==192
assert len({(r['episode'],r['hour'],r['arm']) for r in PROMPTS})==192
assert len({r['prompt_sha256'] for r in PROMPTS})==164  # 28 pairs intentionally byte-identical in stable/ambiguous controls
assert len({r['provider_request_id'] for r in GEMINI})==192
assert all(hashlib.sha256(r['full_prompt'].encode()).hexdigest()==r['prompt_sha256'] for r in PROMPTS)
assert all(r['system_prompt'] and r['temperature']==0.2 and r['truth'] in ('WEST','EAST','ABSTAIN') for r in PROMPTS)
assert {r['kind'] for r in PROMPTS}=={'switch','stable','ambiguous'}
assert {a:sum(r['arm']==a for r in PROMPTS) for a in ARMS}==dict.fromkeys(ARMS,48)
for key in {(r['episode'],r['hour']) for r in PROMPTS}:
    rs=[r for r in PROMPTS if (r['episode'],r['hour'])==key]
    assert len(rs)==4 and {r['arm'] for r in rs}==set(ARMS)
    assert len({json.dumps(json.loads(r['full_prompt'])['observations'],sort_keys=True) for r in rs})==1
# One switch episode, one ambiguous episode, one decision hour each; four arms in each = 8 calls/model.
pilot_ids=sorted({r['episode'] for r in PROMPTS if r['kind']=='switch'})[:1] + sorted({r['episode'] for r in PROMPTS if r['kind']=='ambiguous'})[:1]
PILOT_INDEX=[i for i,p in enumerate(PROMPTS) if p['episode'] in pilot_ids and p['hour']==min(x['hour'] for x in PROMPTS if x['episode']==p['episode'])]
assert len(PILOT_INDEX)==8 and {PROMPTS[i]['arm'] for i in PILOT_INDEX}==set(ARMS)
print('PASS: 192 calls / 164 distinct input texts (28 repeated pairs on stable/ambiguous controls); 48 timepoints; 24 episodes; 4 arms')
print('Pilot indices:',PILOT_INDEX,'| 8 calls per model, 16 total')
print('Selected models:',[x['id'] for x in MODELS.values()])

In [ ]:
# Cell 3 — Load OpenRouter key from Colab Secrets; never print the key
KEY=os.getenv('OPENROUTER_API_KEY','')
if not KEY:
    try:
        from google.colab import userdata
        KEY=userdata.get('OPENROUTER_API_KEY') or ''
    except Exception:
        pass
print('OPENROUTER_API_KEY configured:',bool(KEY))
print('Provider charges depend on the chosen model and token usage; check https://openrouter.ai/ before authorization.')

In [ ]:
# Cell 4 — Runner and parser (defines functions only, NO API calls)
def parse_choice(raw):
    if isinstance(raw,list):
        raw=''.join(x.get('text','') for x in raw if isinstance(x,dict))
    if not isinstance(raw,str):raise ValueError('Assistant message has no textual content')
    raw=raw.strip()
    if raw.startswith('```'):
        raw=re.sub(r'^```(?:json)?\s*','',raw,flags=re.I).split('```',1)[0].strip()
    try:d=json.loads(raw)
    except json.JSONDecodeError:
        a=raw.find('{');b=raw.rfind('}')
        if a<0 or b<=a:raise ValueError('No JSON object in response')
        d=json.loads(raw[a:b+1])
    if not isinstance(d,dict) or d.get('source') not in ('WEST','EAST','ABSTAIN'):
        raise ValueError('source is not WEST, EAST, or ABSTAIN')
    return d

def load_ok(model_key):
    p=ROOT/f'{model_key}_calls.jsonl'
    if not p.exists():return []
    return [json.loads(s) for s in p.read_text().splitlines() if s.strip()]

def run_batch(model_key, indices, authorize=False):
    assert model_key in MODELS
    cfg=MODELS[model_key]
    complete={r['index']:r for r in load_ok(model_key)}
    assert len(complete)==len(load_ok(model_key))
    for idx,r in complete.items():
        assert r['status']=='ok' and r['model_requested']==cfg['id'] and r['prompt_sha256']==PROMPTS[idx]['prompt_sha256']
    waiting=[i for i in indices if i not in complete]
    print(f'{model_key}: {len(waiting)} remaining / {len(indices)} requested; max success calls: 192; new authorization: {authorize}')
    if not authorize:
        print('DRY RUN ONLY: no API calls. Confirm credits and then authorize explicitly.')
        return
    if not KEY:raise RuntimeError('OPENROUTER_API_KEY not configured — run Cell 3')
    path=ROOT/f'{model_key}_calls.jsonl';errors=ROOT/f'{model_key}_errors.jsonl'
    for count,i in enumerate(waiting,1):
        p=PROMPTS[i]
        payload={'model':cfg['id'],'temperature':0.2,'max_tokens':cfg['max_tokens'],
                 'messages':[{'role':'system','content':p['system_prompt']},{'role':'user','content':p['full_prompt']}]}
        req=urllib.request.Request('https://openrouter.ai/api/v1/chat/completions',
            data=json.dumps(payload).encode(),
            headers={'Authorization':'Bearer '+KEY,'Content-Type':'application/json',
                     'HTTP-Referer':'https://colab.research.google.com','X-Title':'WindShift Cross Model Validation'},
            method='POST')
        result={'index':i,'episode':p['episode'],'hour':p['hour'],'kind':p['kind'],'truth':p['truth'],
                'arm':p['arm'],'prompt_sha256':p['prompt_sha256'],'model_requested':cfg['id'],
                'temperature':0.2,'max_tokens':cfg['max_tokens'],
                'time_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
        try:
            with urllib.request.urlopen(req,timeout=120) as resp:response=json.loads(resp.read().decode())
            msg=response['choices'][0]['message']
            parsed=parse_choice(msg.get('content'))
            result.update({'status':'ok','choice':parsed,'raw_response':msg.get('content'),
                           'provider_request_id':response.get('id'),'model_returned':response.get('model'),
                           'usage':response.get('usage',{}),'finish_reason':response['choices'][0].get('finish_reason')})
            if not result['provider_request_id']:
                raise ValueError('Provider response has no request ID; manual audit required')
            with path.open('a') as f:f.write(json.dumps(result,ensure_ascii=False)+'\n')
            if count==1 or count%8==0 or count==len(waiting):
                print(f'{model_key}: {count}/{len(waiting)} newly completed; index={i}; source={parsed["source"]}')
        except Exception as e:
            result.update({'status':'error','error_type':type(e).__name__,'error_message':str(e)[:1000]})
            with errors.open('a') as f:f.write(json.dumps(result,ensure_ascii=False)+'\n')
            print(f'STOP: {model_key} index={i}, {type(e).__name__}: {str(e)[:200]}')
            print('Inspect the error log before rerunning. A provider may charge for unsuccessful attempts.')
            break

assert parse_choice('```json\n{"source":"WEST"}\n```')['source']=='WEST'
assert parse_choice(' {"source":"ABSTAIN", "reason":"uncertain"} ')['source']=='ABSTAIN'
print('Runner and parser initialized. NO inference calls made.')

In [ ]:
# Cell 5 — PILOT: 8 matched calls/model. AUTHORIZATION REQUIRED
AUTHORIZE_PILOT=False   # Change to True only after checking OpenRouter credits
for model_key in ('qwen','claude'):
    run_batch(model_key, PILOT_INDEX, authorize=AUTHORIZE_PILOT)

In [ ]:
# Cell 6 — Inspect pilot before full run
for model_key,cfg in MODELS.items():
    rows=load_ok(model_key)
    print('\nMODEL',cfg['id'],'OK',len(rows),'error log', (ROOT/f'{model_key}_errors.jsonl').exists())
    for r in rows[:8]:print(r['episode'],r['kind'],r['arm'],r['choice']['source'],'truth=',r['truth'], 'usage=',r.get('usage',{}).get('cost'))
    if len(rows)>=8:
        assert all(r['prompt_sha256']==PROMPTS[r['index']]['prompt_sha256'] for r in rows)
        assert len({r['provider_request_id'] for r in rows})==len(rows)
print('Do not proceed if parse errors or model IDs differ. Include adverse results in reporting.')

In [ ]:
# Cell 7 — FULL: authorize remaining calls only after pilot review
AUTHORIZE_FULL=False  # Change to True for remaining calls (max 192 success records per model)
for model_key in ('qwen','claude'):
    run_batch(model_key, list(range(192)), authorize=AUTHORIZE_FULL)

In [ ]:
# Cell 8 — Read-only audit, matched counts and episode bootstrap (NO API calls)
import collections
rows_all={}
rows_all['gpt']=[{'episode':r['episode'],'hour':r['hour'],'kind':r['kind'],'truth':r['truth'],'arm':r['arm'],'choice':r['original_gpt_choice']} for r in GEMINI]
rows_all['gemini']=[{'episode':r['episode'],'hour':r['hour'],'kind':r['kind'],'truth':r['truth'],'arm':r['arm'],'choice':r['choice']['source']} for r in GEMINI]
for k in MODELS:
    rows_all[k]=[{'episode':r['episode'],'hour':r['hour'],'kind':r['kind'],'truth':r['truth'],'arm':r['arm'],'choice':r['choice']['source']} for r in load_ok(k)]
summary={}
for model,rows in rows_all.items():
    condition={}
    for kind in ('switch','stable','ambiguous'):
        for arm in ARMS:
            subset=[r for r in rows if r['kind']==kind and r['arm']==arm]
            condition[kind+'|'+arm]={'n':len(subset),'correct':sum(r['choice']==r['truth'] for r in subset),
                'unnecessary_abstain':sum(r['choice']=='ABSTAIN' and r['truth']!='ABSTAIN' for r in subset),
                'unsupported':sum(r['choice'] not in ('ABSTAIN',r['truth']) for r in subset)}
    summary[model]={'total':len(rows),'conditions':condition}
    print('\n',model,'completed',len(rows),'/192')
    for arm in ARMS:print(arm,condition['switch|'+arm])
(ROOT/'all_model_summary.json').write_text(json.dumps(summary,indent=2))
print('Counts written:',ROOT/'all_model_summary.json')

def cluster_effect(rows,seed=20261107,B=10000):
    subset=[r for r in rows if r['kind']=='switch']
    eps=sorted({r['episode'] for r in subset})
    if len(eps)<16 or len(subset)!=128:return None
    by_ep={e:[r for r in subset if r['episode']==e] for e in eps}
    def effect(sample):
        rr=[r for e in sample for r in by_ep[e]]
        rates={a:sum(r['choice']==r['truth'] for r in rr if r['arm']==a)/sum(r['arm']==a for r in rr) for a in ARMS}
        A,B,C,D=(rates[a] for a in ARMS)
        return {'content':(A+B-C-D)/2,'timestamp':(A+C-B-D)/2,'interaction':A-B-C+D}
    center=effect(eps);rng=random.Random(seed);bs={k:[] for k in center}
    for _ in range(B):
        row=effect([eps[rng.randrange(len(eps))] for _ in eps])
        for k,v in row.items():bs[k].append(v)
    for k in center:bs[k].sort()
    return {k:{'estimate_pp':round(center[k]*100,2),'CI95_pp':[round(bs[k][int(B*.025)]*100,2),round(bs[k][int(B*.975)-1]*100,2)]} for k in center}
contrasts={m:cluster_effect(rs) for m,rs in rows_all.items()}
(ROOT/'episode_bootstrap_effects.json').write_text(json.dumps(contrasts,indent=2))
print('\nClustered contrasts:',json.dumps(contrasts,indent=2))

In [ ]:
# Cell 9 — Export audited results (works after pilot or full study)
from zipfile import ZipFile,ZIP_DEFLATED
out=ROOT/'WindShift_Additional_Model_Results.zip'
notes=(
 '# WindShift additional-model evidence\n\n'
 'This archive contains real OpenRouter calls ONLY if authorized and logged.\n'
 'No mock responses are included. Original GPT/Gemini baselines and exact prompts are frozen.\n'
 'Models: qwen/qwen3-235b-a22b-2507 and anthropic/claude-haiku-5.5.\n'
 'These are reviewer-only follow-ups with GPT-generated peers, not independent full-swarm replications.\n'
 'Any errors are included separately. Use completed counts before interpreting results.\n'
 'Both models were selected before examining their results. No cherry-picking.\n'
)
(ROOT/'README.md').write_text(notes)
with ZipFile(out,'w',ZIP_DEFLATED) as z:
    for name in ['README.md','frozen_reviewer_prompts.json','gemini_reviewer_calls.jsonl',
                 'qwen_calls.jsonl','claude_calls.jsonl','qwen_errors.jsonl','claude_errors.jsonl',
                 'all_model_summary.json','episode_bootstrap_effects.json']:
        pp=ROOT/name
        if pp.exists():z.write(pp,arcname=name)
    z.writestr('manifest_sha256.json',json.dumps({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in ROOT.glob('*.jsonl')},indent=2))
print('Saved',out, 'bytes',out.stat().st_size)
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('Find file at:',out)

## After running

Upload **WindShift_Additional_Model_Results.zip** to the same ChatGPT conversation. The audit will compare **all four model families** with episode-clustered intervals and inspect invalid responses, directional bias and statistical limitations. **Do not claim additional-model results until those provider calls actually run.** If a chosen model is unavailable, stop and report it; do not silently substitute another model after seeing the outputs.